## Polygon Scanline Rasterisation

#### Objective:
- $\text{Find an algorithm based on the Scanline Triangle, to fill any polygon efficiently}$


- $\text{The main idea of the scanline algorithm is to find spans of pixels in one line, that are all inside the shape, so that we can draw them together in one pass}$
#### $\text{How it worked for Triangles:}$
- $\text{We found the left and the right pixels, by rasterising the edges of the triangles} \\ \text{Then filled the span of pixels between them}$

#### $\text{How it worked for general Polygon:}$

<p align="center">
    <img src="image_U11/Screenshot 2025-06-22 at 12.03.44.png" height="400" width="400"/>
    <img src="image_U11/Screenshot 2025-06-22 at 12.05.00.png" height="400" width="400"/>
<p/>

- $\text{It's possible there will be multiple edges intersecting the same scanline}$
- $\text{So when we rasterise the edges, we may find multiple points intersecting each line}$
<p align="center">
    <img src="image_U11/Screenshot 2025-06-22 at 12.07.01.png" height="250" width="250"/>
    <img src="image_U11/Screenshot 2025-06-22 at 12.07.16.png" height="250" width="250"/>
    <img src="image_U11/Screenshot 2025-06-22 at 12.07.22.png" height="250" width="250"/>
    <img src="image_U11/Screenshot 2025-06-22 at 12.07.28.png" height="250" width="250"/>
    <img src="image_U11/Screenshot 2025-06-22 at 12.10.42.png" height="250" width="250"/>
    <img src="image_U11/Screenshot 2025-06-22 at 12.11.04.png" height="250" width="250"/>
<p/>



#### Solving Scanline for General polygons

1. $\text{How do we know which edges to rasterise in each line}$
- $\text{A. Find the intersection of the edges with scanlines of pixels}$
2. $\text{How can find spans (lines) of pixels to draw}$
- $\text{A. Fill them using the odd-pairty rule:}$
- $\text{You start drawing with the first intersection in the line and stop with the next intersection of the line}$
- $\text{Skip to the next intersection of the line to start rasterising again and then stop at the proceeding intersection etc}$

<p align="center">
    <img src="image_U11/Screenshot 2025-06-22 at 12.15.38.png" height="200" width="400"/>
    <img src="image_U11/Screenshot 2025-06-22 at 12.22.15.png" height="200" width="400"/>
    <img src="image_U11/Screenshot 2025-06-22 at 12.27.35.png" height="200" width="400"/>
<p/>

```
ScanConvertPolygon (Polygon P, Color c)
{
    For y = P -> yMin to P -> yMax do {
        XList = intersect all edges of P with line y;
        Sort XList increase x order: 
        Fill pixels in line y with alternating segements of XList with color c;
    }
}
```





### How to implement efficiently

- $\text{We create two lists of edges:}$
1. $\text{The rasterised edge list - Initialised empty}$
2. $\text{The potential edge list - Initialised containing all edges of the polygon}$

- $\text{We sort the potential edge list increasing order of min y-value (as we're starting from the top to the bottom)}$
- $\text{Whenever, we advance to the next scan line we check if edges in the potential edge list should move to the rasterised edge list}$
- $\text{We do this by examining their min-y-value} \\ \text{checking the first edge we ask if the min-y value is larger than the current scanline y -value, then we stop the search}$
$\text{All other edge in the list would have a larger minimal y-value}$
- $\text{The minimum y-value in the potential list is smaller or equal to the current y-value, we must move it to the rasterized edge list} \\ \text{continue iterating the potential rasterized list}$
- $\text{This way we maintain an order where we only need to rasterize a span of pixels each scanline once}$
- $\text{In the rasterized edge list we sort the edges by their max-y value, since we know if the max y-value of the edge is smaller than the current scanline} \\ \text{the edge must be removed from the list}$
- $\text{Once we've done this proceedure, to rasterize becomes simple, we go over the edge in the razsterized list, and find all the pixels of the intersection of the edges}$
$\text{with the current scanline}$
- $\text{Once we find the pixels, we sort by x-values}$

```cpp
ScanConvertPolygon(Polygon P, Color c)
{
    // Step 1: Build and sort the Edge Table (ET) by increasing y_min
    EdgeTable ET = BuildSortedEdgeTable(P); 

    // Step 2: Initialize the Active Edge Table (AET)
    ActiveEdgeTable AET = empty;

    // Step 3: Loop through each scanline (from top to bottom)
    for (int y = ET.y_min to ET.y_max) {
        
        // Step 3a: Remove edges from AET where y == y_max
        AET.removeEdgesEndingAt(y);

        // Step 3b: Add to AET edges from ET that start at y (i.e., y == y_min)
        AET.addEdgesStartingAt(y, ET);

        // Step 3c: Sort AET by current x intersection values
        AET.sortByX();

        // Step 3d: Fill between pairs of x-intersections
        for (int i = 0; i < AET.size(); i += 2) {
            int x_start = AET[i].x;
            int x_end   = AET[i+1].x;
            SetPixelRange(x_start, x_end, y, c);
        }

        // Step 3e: Update x-intersections for next scanline using slope (1/m)
        AET.updateXValues();
    }
}
```

- $\text{Complexity: O(klogk +N)} \\ \text{where K = Number of  edges} \\ \text{N = Number of pixels}$



### Running Concrete Example:
- $\text{Consider the polygon below where we denote the edges by } e_i$
- $\text{We've already pre-processed the edges (via line segment rasterisation)} \\ \text{this means, we have a linked list of array, where each array segement is ordered from min y-value to max y values }$
$\text{So if we have 8 edges we'll have 8 array segments in the linked list, and this whole LL is sorted by min starting value}$
1. $\text{Starting at y = 0 (the top of the polygon) we increment the y value until y = first value in the linked-list }$
$\text{In the example we have edges 8 and 7}$
2. $\text{We place the respective edges in the Rasterized list, in this list we sort by MAX y-value} \\ \text{This means that edge 7 is before edge 8 }$
3. $\text{Whilst iterating through the y-values in increments of 1, we also check rasterized list if there's a match with theor max values} \\ \text{This will indicate to remove the respective edge from the list}$

|Step | Action | Explanantion |
|-----|--------|--------------|
|<img src="image_U11/Screenshot 2025-06-22 at 12.49.19.png" height="250" width="250"/>| E7 and E8 are added to the list <br> and at the current line are being rasterised| |
| <img src="image_U11/Screenshot 2025-06-22 at 12.49.28.png" height="250" width="250"/>|Color between E7-E8 and E6-E5 |At each new line we check both list <br> and iterate between pair edges|
| <img src="image_U11/Screenshot 2025-06-22 at 12.49.41.png" height="250" width="250"/>|Add E3 and E2 to AET | Note that edge pairs are only at one list at a time|
| <img src="image_U11/Screenshot 2025-06-22 at 12.49.54.png" height="250" width="250"/>|Remove edge E7, E6, E5 from EAT and also add E4 to EAT| We sort by y-max of an edge <br> min y-max to max y-max|
| <img src="image_U11/Screenshot 2025-06-22 at 12.50.13.png" height="250" width="250"/>|Remove E3-E4 |Once the ET is empty it remains empty | 
| <img src="image_U11/Screenshot 2025-06-22 at 12.50.21.png" height="250" width="250"/>|Rasterise remainder| | 


### Flood Fill vs Scan Line
| $\textbf{Flood Fill}$                          | $\textbf{ScanLine}$                   |
|------------------------------------------------|---------------------------------------|
|$\text{Simple to Implement}$                    | $\text{More complex to Implement}$    |
|$\text{Requires a seed point}$                  | $\text{No Seed point needed}$         |
|$\text{Requires very large stack size}$         | $\text{Requires small stack size}$    |
|$\text{Common in paint packages}$               | $\text{Used in rendering (Efficient)}$|


### Aliasing
$\text{Aliasing is a potential consequence that could occur in the rasterisation process in the Graphical Pipeline}$

$\text{As mentioned in Unit-1, to deal with aliasing was by smoothing the image}$

$\text{Since we've created the image and populated the pixels, we can improve the process}$

### Anti-Aliasing (Higher Sample Rate)

- $\text{In the first rendering algorithm (Ray-Tracing), we handles anti-aliasing by shooting more rays into each pixel, and averaging the results}$
- $\text{We render an image with more pixels than needed (higher sampling rate) and then average them}$
- $\text{We could obtain an image which is 2x the desired size and then obtain 4x the number of pixels needed, and then average the values of every 4 pixels, to obatin an anti-aliased pixel}$
- $\text{The cost to do this is the rendering time}$

<p align="center">
    <img src="image_U11/Screenshot 2025-06-22 at 12.59.14.png" height="300" width="300"/>
    <img src="image_U11/Screenshot 2025-06-22 at 13.02.22.png" height="300" width="300"/>
    <img src="image_U11/Screenshot 2025-06-22 at 13.02.30.png" height="300" width="300"/>
    <img src="image_U11/Screenshot 2025-06-22 at 13.02.38.png" height="300" width="300"/>
<p/>
